In [1]:
from constants import DATA_ROOT_PATH_NAME, BANDPASS, HAMPEL_WINDOW_SIZE, HAMPEL_N_SIGMA, CROP_TMIN, CROP_TMAX, LOCAL_DETREND_WINDOW_SEC, LOCAL_DETREND_STEP_SEC, ASR_CUTOFF, ASR_BLOCKSIZE, ASR_WIN_LEN, ASR_WIN_OVERLAP, ASR_MAX_DROPOUT_FRACTION, ASR_MIN_CLEAN_FRACTION, ASR_MAX_BAD_CHANS

from preprocessing.step.bandpass import BandpassFilterStep
from preprocessing.step.detrend import LocalDetrendStep
from preprocessing.step.hampel import HampelFilterStep
from preprocessing.step.asr import ASRStep
from preprocessing.step.crop import CropStep

from preprocessing.pipeline import PreprocessingPipeline
import numpy as np

from features.factory import FeatureExtractionEngine, FeatureExtractionConfig
from features.categories import FeatureCategory

from eeg.data import EEGRecordedDataProvider

from features.visualization import ExtractedFeatureHeatmapFactory



%load_ext autoreload
%autoreload 2

# Création des 2 datasets de features

In [2]:
recordings = EEGRecordedDataProvider.build(DATA_ROOT_PATH_NAME)

/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-eyesclosed.

The search_str was "data/sub-001/**/eeg/sub-001*events.tsv"
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 57
Group: A
MMSE: 16
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not find any events.tsv associated with sub-002_task-eyesclosed.

The search_str was "data/sub-002/**/eeg/sub-002*events.tsv"
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 78
Group: A
MMSE: 22
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not fin

In [3]:
asr_pipeline = PreprocessingPipeline(name="ASR",
                                        steps=[
                                                BandpassFilterStep(BANDPASS),
                                                CropStep(tmin=CROP_TMIN, tmax=CROP_TMAX),
                                                ASRStep(cutoff=ASR_CUTOFF, blocksize=ASR_BLOCKSIZE, win_len=ASR_WIN_LEN, win_overlap=ASR_WIN_OVERLAP, max_dropout_fraction=ASR_MAX_DROPOUT_FRACTION, min_clean_fraction=ASR_MIN_CLEAN_FRACTION, max_bad_chans=ASR_MAX_BAD_CHANS)
                                                ])

dethamp_pipeline = PreprocessingPipeline(name="det-hamp",
                                        steps=[ 
                                                BandpassFilterStep(BANDPASS),
                                                CropStep(tmin=CROP_TMIN, tmax=CROP_TMAX),
                                                LocalDetrendStep(window_sec=LOCAL_DETREND_WINDOW_SEC, step_sec=LOCAL_DETREND_STEP_SEC),
                                                HampelFilterStep(window_size=HAMPEL_WINDOW_SIZE, n_sigma=HAMPEL_N_SIGMA)
                                                ])

In [4]:
recorded_eeg = recordings[14]
asr_processed_eeg = asr_pipeline.compute(recorded_eeg)
dethamp_processed_eeg = dethamp_pipeline.compute(recorded_eeg)

In [5]:
categories_to_extract = [FeatureCategory.WAVELET, FeatureCategory.TEMPORAL, FeatureCategory.POWER_RATIO, FeatureCategory.SPECTRAL]
config = FeatureExtractionConfig(categories_to_extract=categories_to_extract, wamp_threshold=10e-9, ppc_epoch_duration=2)
feature_extraction_engine = FeatureExtractionEngine(config=config)

dethamp_processed_extraction_result = feature_extraction_engine.extract(dethamp_processed_eeg)
dethamp_processed_extracted_feature_heatmap_factory = ExtractedFeatureHeatmapFactory(dethamp_processed_extraction_result)

asr_processed_extraction_result = feature_extraction_engine.extract(asr_processed_eeg)
asr_processed_extracted_feature_heatmap_factory = ExtractedFeatureHeatmapFactory(asr_processed_extraction_result)

/mnt/ssd2/pth-eeg/eeg/eeg/ppc.py:395: RuntimeWarning: fmin=1.000 Hz corresponds to 2.000 < 5 cycles based on the epoch length 2.000 sec, need at least 5.000 sec epochs or fmin=2.500. Spectrum estimate will be unreliable.
  conn: Connectivity = spectral_connectivity_epochs(
/mnt/ssd2/pth-eeg/eeg/eeg/ppc.py:395: RuntimeWarning: fmin=1.000 Hz corresponds to 2.000 < 5 cycles based on the epoch length 2.000 sec, need at least 5.000 sec epochs or fmin=2.500. Spectrum estimate will be unreliable.
  conn: Connectivity = spectral_connectivity_epochs(


In [6]:
from features.io import FeaturesDatasetIO
dataset = FeaturesDatasetIO.load("computed_data/dethamp")


In [7]:
dataset.to_wide_dataframe()

,subject_id,subject_health,Fp1_theta_beta_ratio,Fp1_theta_alpha_ratio,Fp1_gamma_alpha_ratio,Fp1_spectral_power_ratio,Fp1_alpha_dominant_frequency,Fp1_gamma_dominant_frequency,Fp1_spectral_rolloff,Fp1_spectral_centroid,...,Pz_crest_factor,Pz_clearance_factor,Pz_willison_amplitude,Pz_zero_crossing_rate,Pz_wavelet_energy_approximate,Pz_wavelet_energy_detail,Pz_relative_wavelet_energy,Pz_wavelet_packet_energy_approximate,Pz_wavelet_packet_energy_detail,Pz_relative_wavelet_packet_energy
0,082,Frontotemporal Dementia,0.008586,0.065275,4.122641,87.131668,12.516249,44.115196,50.748308,38.982905,...,3.281268,4.644750,29935.0,0.184027,6.190461e-07,4.964458e-08,0.925759,6.190461e-07,4.964458e-08,0.925759
1,003,Alzheimer,0.013065,0.057561,1.973654,88.581747,12.516249,38.098730,50.181661,41.203296,...,3.873913,5.555689,29926.0,0.182261,5.997947e-07,4.789610e-08,0.926051,5.997947e-07,4.789610e-08,0.926051
2,062,Healthy,0.003463,0.066402,16.052994,194.642301,12.499583,41.098630,50.431652,39.837350,...,2.959246,3.857358,29973.0,0.199360,1.761156e-06,1.729124e-07,0.910597,1.761156e-06,1.729124e-07,0.910597
3,067,Frontotemporal Dementia,0.005972,0.126334,22.157517,102.214283,12.499583,34.165528,52.798240,39.590599,...,3.781434,5.318567,29927.0,0.190660,9.704436e-07,8.466753e-08,0.919755,9.704436e-07,8.466753e-08,0.919755
4,086,Frontotemporal Dementia,0.012788,0.287117,33.158656,51.933039,12.299590,38.332056,101.146628,42.354078,...,4.959957,8.452008,29433.0,0.152928,8.644196e-07,9.589875e-08,0.900139,8.644196e-07,9.589875e-08,0.900139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,059,Healthy,0.002989,0.104133,23.949990,217.078013,12.532916,39.148695,51.214960,33.850325,...,3.347949,4.787314,29917.0,0.178661,4.996948e-07,3.613499e-08,0.932563,4.996948e-07,3.613499e-08,0.932563
84,046,Healthy,0.007188,0.059028,2.455603,153.880253,12.516249,42.431919,49.998333,38.428572,...,3.413036,4.965457,29932.0,0.166328,1.057191e-06,7.014353e-08,0.937779,1.057191e-06,7.014353e-08,0.937779
85,028,Alzheimer,0.013847,0.114711,4.141440,71.554850,12.516249,33.415553,51.748275,36.418479,...,3.224689,4.577200,29925.0,0.182427,5.184591e-07,4.012528e-08,0.928166,5.184591e-07,4.012528e-08,0.928166
86,080,Frontotemporal Dementia,0.009805,0.063023,3.763163,99.382980,12.499583,33.732209,50.831639,38.482353,...,3.272357,4.584478,29851.0,0.190394,1.031599e-06,8.553544e-08,0.923433,1.031599e-06,8.553544e-08,0.923433
